In [1]:
import numpy
import sys
import os
import matplotlib.pyplot as plt
# import skimage
import nrrd
from scipy import ndimage
import meshio
from skimage.morphology import skeletonize
from scipy.interpolate import splprep, splev
from sklearn.decomposition import PCA
from scipy.spatial import cKDTree
from scipy.interpolate import UnivariateSpline
import trimesh
from scipy.ndimage import median_filter

from scipy.spatial import cKDTree
from scipy.interpolate import UnivariateSpline
from skimage import measure
import tomllib
from trimesh.path.entities import Line

import pyvista as pv

from scipy.ndimage import uniform_filter1d

sys.path.insert(0, "../src")  # add Folder_2 path to search list

from meshing_functions import getSurfaceMesh, export_centerline,  tetra_mesh_from_stl, set_the_offset, tetra_shell_from_two_surfaces
from image_processing_functions import load_vti
from extended_muscle_meshing import (optic_nerve_tube, remesh_tube_as_solid, cut_by_eyeball,
                                     gmsh_mesh_from_surface, mesh_report)
from centerlines import fit_analytical_centerline

with open("/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/config.toml", mode="rb") as f:
    config = tomllib.load(f)

sequence = "T1"

if sequence == "T2":
    base            = config["data"]["base"]
    input_file      = base + config["data"]["nii_file_T2"]
    output_folder   = config["data"]["output_folder_ext"]
    segment_filename = config["data"]["segment_filename_T2"]
    segment_folder  = config["data"]["segment_folder_T2"]
    centerline_folder = config["data"]["centerline_folder"]

else:
    base            = config["data"]["base"]
    input_file      = base + config["data"]["nii_file"]
    output_folder   = config["data"]["output_folder_ext"]
    segment_filename = config["data"]["segment_filename"]
    segment_folder  = config["data"]["segment_folder"]
    centerline_folder = config["data"]["centerline_folder"]

export_name = base + output_folder + segment_filename[:-8]

eye_mesh = trimesh.load(base + output_folder +"re-meshed_adaptive/id_2_shell__h_0-4.stl")



## Load image parameters

In [2]:
data, header = nrrd.read(base + segment_folder + segment_filename)

dirs = [numpy.array(d, dtype=float) for d in header["space directions"] if d is not None and not numpy.any(numpy.isnan(numpy.array(d, dtype=float)))]
dirs = numpy.array(dirs)

pixel_spacing = numpy.linalg.norm(dirs, axis=1)           # mm per voxel, in array axis order
direction = (dirs / pixel_spacing[:, None]).T          # columns = unit axis directions
origin = numpy.array(header["space origin"], dtype=float)

print("original pixel spacing = ", pixel_spacing)
print("segmentation origin = ", origin)

original pixel spacing =  [0.3125     0.3125     0.50000006]
segmentation origin =  [-55.95619965 -90.20069885  20.49090004]


In [3]:

n_theta_points = 40

voxel_mm = float(numpy.min(pixel_spacing))   # isotropic spacing after resampling
contact_axis_1 = 8 * voxel_mm
contact_axis_2 = 3 * voxel_mm
print("contact semi-axes (mm):", contact_axis_1, contact_axis_2)

contact semi-axes (mm): 2.499999999999999 0.9374999999999997


## Load segments


In [4]:
segments = []
i = 0
while f"Segment{i}_Name" in header:
    name  = header[f"Segment{i}_Name"]
    label = int(header.get(f"Segment{i}_LabelValue", i + 1))
    layer = int(header.get(f"Segment{i}_Layer", 0))
    vol   = data[layer] if data.ndim == 4 else data
    segments.append((name, vol == label))
    print(f"{name}: layer={layer}, value={label}, voxels={segments[-1][1].sum()}")
    i += 1

combined = numpy.zeros(segments[0][1].shape, dtype=numpy.uint8)
for new_id, (name, mask) in enumerate(segments, start=1):
    combined[mask] = new_id          # later segments overwrite earlier ones where they overlap

data = combined
id_max = int(data.max())


optical nerve: layer=0, value=1, voxels=15624
eyeball: layer=0, value=3, voxels=160561
lateral rectus: layer=0, value=4, voxels=14436
medial rectus: layer=0, value=5, voxels=13164
inferior rectus: layer=0, value=6, voxels=11961
superior rectus: layer=0, value=7, voxels=13568
lens: layer=0, value=2, voxels=1508


## Resample

In [5]:
pixel_spacing = numpy.asarray(pixel_spacing, dtype=float)
target = pixel_spacing.min()

if not numpy.allclose(pixel_spacing, target):
    factors = pixel_spacing / target
    data = ndimage.zoom(data, factors, order=0)
    print("resampling factors = ", factors)
    print("new dimensions = ", data.shape)

pixel_spacing = target

print("Final uniform pixel_spacing = ", pixel_spacing)

resampling factors =  [1.         1.         1.60000019]
new dimensions =  (145, 209, 162)
Final uniform pixel_spacing =  0.3124999999999999


## Define IDs

In [6]:
eyeball_id = 2
lens_id = 7

muscle_ids = [3,4]
ON_id = 1



## Load masks

In [7]:
muslce_masks = []

for id in muscle_ids: 
    mask = numpy.zeros(data.shape, dtype=numpy.uint8)
    mask[(data>id-0.5) & (data<id+0.5 )]=1
    muslce_masks.append(mask)

ON_mask = numpy.zeros(data.shape, dtype=numpy.uint8)
ON_mask[(data>ON_id-0.5) & (data<ON_id+0.5 )]=1

eyeball_mask = numpy.zeros(data.shape, dtype=numpy.uint8)
eyeball_mask[(data>eyeball_id-0.5) & (data<eyeball_id+0.5 )]=1


# Part II

In [8]:
def get_local_frame(direction):
    """Calculates two vectors perpendicular to the spine direction."""
    # Pick a non-parallel vector to start the cross product
    # vec = numpy.array([1, 0, 0]) if abs(direction[0]) < 0.95 else numpy.array([0, 1, 0])

    vec_1 = numpy.array([1, 1, 0])
    vec = vec_1/numpy.linalg.norm(vec_1)

    v1 = numpy.cross(direction, vec)
    v1 /= numpy.linalg.norm(v1)
    v2 = numpy.cross(direction, v1)
    return v1, v2


## Find intersection of the extended centerline and the eyeball


In [9]:
def find_eye_intersection_index(centerline_extended, mask_eye):
    """
    Returns the index along the extended centerline that intersects the eye.
    """
    for idx, point in enumerate(centerline_extended):
        z, y, x = numpy.round(point).astype(int)
        
        # Check bounds
        if 0 <= z < mask_eye.shape[0] and 0 <= y < mask_eye.shape[1] and 0 <= x < mask_eye.shape[2]:
            if mask_eye[z, y, x] == 1:
                return idx  
                
    return len(centerline_extended) - 1


In [10]:
def compute_local_tangent_from_section(section_2d_path, contact_point_mm):
    """
    Computes the normalized tangent vector using the topological 
    line entities of the trimesh section.
    """
    vertices = section_2d_path.vertices
    
    # 1. Find the entity segment closest to the contact point
    min_dist = float('inf')
    best_tangent = None
    
    for entity in section_2d_path.entities:
        # Entities contain ordered point indices, e.g. [idx_a, idx_b]
        points = entity.points
        for i in range(len(points) - 1):
            pA = vertices[points[i]]
            pB = vertices[points[i + 1]]
            
            # Midpoint of segment
            seg_mid = (pA + pB) / 2.0
            dist = numpy.linalg.norm(seg_mid - contact_point_mm)
            
            if dist < min_dist:
                min_dist = dist
                vec = pB - pA
                norm_vec = numpy.linalg.norm(vec)
                if norm_vec > 1e-8:
                    best_tangent = vec / norm_vec

    if best_tangent is None:
        raise ValueError("Could not extract a valid tangent from the section curve.")

    return best_tangent

## Set meshing parameters

In [11]:
n_theta_points = 40
smoothing_factor = 15
spline_order = 3

## Get contact points & surfaces

In [12]:
def load_section(path):
    pts = numpy.load(path)                                  # (n, 3)
    return trimesh.path.Path3D(entities=[Line(numpy.arange(len(pts)))], vertices=pts)


## Centerline

In [13]:

target_point = numpy.array(ndimage.center_of_mass(eyeball_mask))   # in voxels
ignored_nodes = 0.05
order = 3


centerline, centerline_ext = fit_analytical_centerline(ON_mask, [target_point], ignored_nodes, order, n_bins=10, extension_factor=0.15)

# eye_idx = find_eye_intersection_index(centerline_ext, eyeball_mask)
# centerline_trimmed = centerline_ext[:eye_idx + 1]

export_centerline(centerline_ext, pixel_spacing, base + centerline_folder + "centerline_ON_exttended.vtp", [0,0,0])
# export_centerline(centerline_trimmed, pixel_spacing, base + centerline_folder + "centerline_ON_trimmed.vtp", [0,0,0])

reverse


## Meshing loop

In [14]:

h = 0.9
h_eyeball = 0.4

tube, info = optic_nerve_tube(
    centerline_ext, ON_mask, eyeball_mask, voxel_mm, n_theta=n_theta_points,
    unreliable_mm=3.0,
    ref_window_mm=3.0,
    flare_length_mm=3.0,
    flare_amount_mm=1.2,
    flare_power=2.0,
    overlap_mm=5)

solid = remesh_tube_as_solid(
    tube, n_theta_points, voxel_mm, resolution_mm=0.1,
    eye_surface=eye_mesh,
    eye_clearance_mm=-0.05,       # keep a thin layer inside the eye, so the exact cut below makes the face
    contact_fill_mm=0.3,          # turn the knife edge into a fillet
    contact_open_mm=0.3)
on_surface = cut_by_eyeball(solid, eye_mesh)   # exact contact face, as before

output_filename = base + output_folder + "re-meshed_adaptive/" + "remeshed_h_" + str(h).replace(".", "-") + "_id_ON"
surf, quality = gmsh_mesh_from_surface(
    on_surface, output_filename, element_size=h,
    refine_near=eye_mesh,          # distance measured to the eyeball surface
    h_near=h_eyeball,              # element size at the contact
    near_distance_mm=0.3,          # keep h_eyeball within 0.3 mm of the eye
    transition_mm=3.0)             # then grow to h over 3 mm


   ON: 122 nodes (19 added towards/into the eye), reliable radius 3.53 mm, at the eye 4.73 mm, contact angle ~51 deg
   contact fill: 7.05 mm^3 added between muscle and eye
   gmsh: remeshed surface, angle 60.0
   edge length: near the surface 0.382 mm (target 0.4), far 0.910 mm (target 0.9); edge/target 5-95 %: 0.66 - 1.39
   triangles: 5072, gamma min 0.315, mean 0.951, < 0.1: 0
   tets: 16714, gamma min 0.316, mean 0.832, < 0.1: 0


In [ ]:
from extended_muscle_meshing import conform_contact_to_surface

# ---------------------------------------------------------------------------
# Conforming contact: the ON's contact face uses the eyeball's own triangles
# ---------------------------------------------------------------------------
improved, contact_info = conform_contact_to_surface(
    surf, eye_mesh,
    rim_tol_mm=1.5*h_eyeball)        # wall triangles within 0.1 mm of the eye count as contact (removes flat elements)   
improved.export(f"{export_name}_ON_improved_contact.stl")

# tet mesh, keeping this surface exactly (same adaptive sizing as before)
output_filename = base + output_folder + "re-meshed_adaptive/" + "remeshed_h_" + str(h).replace(".", "-") + "_id_ON_improved_contact"
surf_ic, quality_ic = gmsh_mesh_from_surface(
    improved, output_filename, element_size=h,
    refine_near=eye_mesh, h_near=h_eyeball,
    near_distance_mm=0.3, transition_mm=20.0,
    keep_surface=True)

# indices into the eyeball mesh of the shared contact nodes / faces, for coupling in the simulation
numpy.save(output_filename + "_eyeball_contact_vertex_ids.npy", contact_info["target_vertex_ids"])
numpy.save(output_filename + "_eyeball_contact_face_ids.npy", contact_info["target_face_ids"])

   conforming contact: 1150 eyeball faces / 618 nodes (74.38 mm^2), patch 'inside, wall kept', 179 stitching triangles (lift rings 0), watertight True
   gmsh: input surface kept
   edge length: near the surface 0.430 mm (target 0.4), far 0.884 mm (target 0.9); edge/target 5-95 %: 0.67 - 1.36
   triangles: 4702, gamma min 0.214, mean 0.940, < 0.1: 0
   tets: 17902, gamma min 0.153, mean 0.833, < 0.1: 0
